# 02 · Do problema de Markowitz ao QUBO: teoria e verificação

**Pergunta-guia:** como transformar a escolha de uma carteira em um problema que um computador quântico (ou um simulador) possa resolver?

Este notebook:

1. Parte da formulação **binária com restrição de orçamento** usada no notebook de referência (*Quantum_Finance_DualQ*, seção "Otimização Revisitada – Abordagem restrita binária").
2. Aplica a formulação aos **4 ativos do projeto** (ITUB4, VALE3, KLBN4, BBAS3), usando apenas o período de **treino (2023–2024)**.
3. Converte a restrição em **penalidade** e mostra que a forma penalizada é exatamente um **QUBO** (matriz quadrática sobre variáveis binárias).
4. Converte o QUBO para o **Hamiltoniano de Ising** (variáveis $z_i \in \{-1, +1\}$), que é a forma usada por QAOA e VQE.
5. Verifica tudo por **força bruta** sobre os 16 vetores possíveis, sem nenhum algoritmo quântico.

Nada aqui depende de Qiskit. O objetivo é entender a formulação antes de escolher ferramentas.

> Referência: *Quantum_Finance_DualQ.ipynb*, célula 58 (formulação) e células 61–64 (parâmetros e operador). Os dados, o período e a verificação são deste projeto.

## 1. O problema

Seja $x \in \{0,1\}^n$, com $n = 4$ ativos. $x_i = 1$ significa "o ativo $i$ entra na carteira". O problema é:

$$
\min_{x \in \{0,1\}^n} \; q\, x^T \Sigma x - \mu^T x
\qquad \text{com a restrição} \qquad \mathbf{1}^T x = B
$$

| Símbolo | Significado | Neste notebook |
|---|---|---|
| $x$ | seleção binária dos ativos | 4 variáveis |
| $\mu$ | retorno esperado anual de cada ativo | média diária × 252 |
| $\Sigma$ | covariância anual entre os ativos | covariância diária × 252 |
| $q$ | apetite ao risco (maior $q$, mais peso ao risco) | $q = 0{,}5$ |
| $B$ | orçamento: quantos ativos escolher | $B = 2$ |

**Por que é diferente de Markowitz clássico?** Em Markowitz os pesos $w$ são contínuos (somam 1, cada um entre 0 e 1). Aqui cada ativo entra **ou não entra**, com peso igual. Essa é a simplificação da DualQ: todos os ativos têm preço normalizado e a carteira tem exatamente $B$ ativos. Perdemos a alocação fina e ganhamos uma forma que um computador quântico consegue representar com 1 qubit por ativo.

**Unidades:** $\mu$ e $\Sigma$ precisam estar na mesma escala de tempo. Aqui ambos são anuais. No notebook 01 a volatilidade era anualizada dentro do loop e o retorno vinha de outra forma; neste notebook a conversão é explícita.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product

np.set_printoptions(precision=4, suppress=True)

# Caminho relativo a esta pasta (notebook/). O CSV tem duas linhas de cabeçalho e uma linha extra ("Date").
df = pd.read_csv('../data/precos_b3.csv', header=[0, 1], index_col=0, skiprows=[2], parse_dates=True)
close = df['Close'].apply(pd.to_numeric, errors='coerce').dropna()
close.index = pd.to_datetime(close.index)

ativos = list(close.columns)
print('Ativos (ordem das colunas):', ativos)
print('Período completo:', close.index.min().date(), 'a', close.index.max().date(), '| observações:', len(close))

Ativos (ordem das colunas): ['BBAS3.SA', 'ITUB4.SA', 'KLBN4.SA', 'VALE3.SA']
Período completo: 2023-01-02 a 2025-12-30 | observações: 749


## 2. Entradas do modelo: μ e Σ no período de treino

Para evitar olhar o futuro, μ e Σ vêm só de **2023–2024**. O período de 2025 fica reservado para teste (como no notebook 01, seção de teste fora da amostra).

In [3]:
TREINO = slice('2023-01-01', '2024-12-31')
retornos_treino = close.loc[TREINO].pct_change().dropna()

mu = retornos_treino.mean() * 252      # retorno esperado anual
Sigma = retornos_treino.cov() * 252    # covariância anual

print('Retorno esperado anual (mu):')
print(mu.round(4).to_string())
print('\nVolatilidade anual:')
print(pd.Series(np.sqrt(np.diag(Sigma)), index=ativos).round(4).to_string())
print('\nCorrelação:')
print(pd.DataFrame(Sigma.values / np.outer(np.sqrt(np.diag(Sigma)), np.sqrt(np.diag(Sigma))),
                   index=ativos, columns=ativos).round(2).to_string())

Retorno esperado anual (mu):
Ticker
BBAS3.SA    0.2946
ITUB4.SA    0.1956
KLBN4.SA    0.2037
VALE3.SA   -0.1331

Volatilidade anual:
BBAS3.SA    0.2100
ITUB4.SA    0.2104
KLBN4.SA    0.2124
VALE3.SA    0.2483

Correlação:
          BBAS3.SA  ITUB4.SA  KLBN4.SA  VALE3.SA
BBAS3.SA      1.00      0.60      0.16      0.13
ITUB4.SA      0.60      1.00      0.16      0.20
KLBN4.SA      0.16      0.16      1.00      0.22
VALE3.SA      0.13      0.20      0.22      1.00


Com dois anos de dados, as estimativas de $\mu$ são ruidosas. Isso não muda a formulação, mas muda o quanto podemos confiar na solução. Voltamos a isso no fim.

## 3. Enumerando todas as possibilidades

Com $n = 4$, existem $2^4 = 16$ vetores binários. Desses, só os que têm exatamente $B = 2$ ativos respeitam a restrição, ou seja $\binom{4}{2} = 6$.

Antes de qualquer penalidade, vamos ver a função objetivo sem restrição, para cada vetor.

In [4]:
n = len(ativos)
B = 2          # orçamento: quantos ativos escolher
q = 0.5        # apetite ao risco

todos = np.array(list(product([0, 1], repeat=n)))   # 16 vetores binários

def objetivo(x, q=q):
    """q * xᵀΣx − μᵀx  (sem penalidade)."""
    return q * x @ Sigma.values @ x - mu.values @ x

tab = pd.DataFrame({
    'x': [''.join(map(str, x)) for x in todos],
    'nº ativos': todos.sum(axis=1),
    'viável': todos.sum(axis=1) == B,
    'f(x)': [objetivo(x) for x in todos],
}).sort_values('f(x)').reset_index(drop=True)

print(f'Total de vetores: {len(todos)} | viáveis (nº ativos = {B}): {tab["viável"].sum()}\n')
tab.head(10)

Total de vetores: 16 | viáveis (nº ativos = 2): 6



,x,nº ativos,viável,f(x)
0,1110,3,False,-0.585972
1,1010,2,True,-0.446466
2,1100,2,True,-0.419401
3,1111,4,False,-0.393288
4,0110,2,True,-0.347257
5,1000,1,False,-0.272554
6,1011,3,False,-0.264113
7,1101,3,False,-0.238222
8,0010,1,False,-0.181147
9,0100,1,False,-0.173451


A tabela mostra o problema sem restrição. Vetores com mais ativos podem ter melhor $f(x)$ porque acumulam mais retorno, e nada impede isso. Se a primeira linha da tabela tiver `viável = False`, o melhor vetor da conta sem restrição não é uma carteira válida. A restrição precisa entrar de algum modo.

## 4. A restrição vira penalidade

A DualQ propõe incluir a restrição na função objetivo como um termo de penalidade:

$$
f_\lambda(x) = q\, x^T \Sigma x - \mu^T x + \lambda\,(\mathbf{1}^T x - B)^2
$$

Se $\lambda > 0$ for grande o bastante, violar a restrição custa mais do que qualquer ganho de retorno. Então o mínimo de $f_\lambda$ cai nas soluções viáveis.

A escolha de $\lambda$ é o ponto delicado: pequeno demais e a penalidade não funciona; grande demais e ela domina o objetivo e esconde o retorno. Testamos isso abaixo.

In [5]:
def f_penalizada(x, lam):
    """f_λ(x) = objetivo(x) + λ (Σx_i − B)²."""
    return objetivo(x) + lam * (x.sum() - B) ** 2

print('Varredura de λ: qual vetor é o mínimo de f_λ, e ele é viável?\n')
for lam_t in [0.001, 0.01, 0.1, 1.0, 10.0]:
    best_x, best_v = min(((x, f_penalizada(x, lam_t)) for x in todos), key=lambda t: t[1])
    print(f'λ = {lam_t:<6} x* = {best_x}  nº ativos = {best_x.sum()}  viável = {best_x.sum() == B}  f = {best_v:.4f}')

Varredura de λ: qual vetor é o mínimo de f_λ, e ele é viável?

λ = 0.001  x* = [1 1 1 0]  nº ativos = 3  viável = False  f = -0.5850
λ = 0.01   x* = [1 1 1 0]  nº ativos = 3  viável = False  f = -0.5760
λ = 0.1    x* = [1 1 1 0]  nº ativos = 3  viável = False  f = -0.4860
λ = 1.0    x* = [1 0 1 0]  nº ativos = 2  viável = True  f = -0.4465
λ = 10.0   x* = [1 0 1 0]  nº ativos = 2  viável = True  f = -0.4465


Com $\lambda$ pequeno, o mínimo tem mais de 2 ativos: a penalidade não foi suficiente. Com $\lambda$ grande, o mínimo é viável. O limiar depende da escala de $\mu$ e $\Sigma$, por isso a escolha de $\lambda$ não pode ser um número mágico. Ela precisa ser testada (é a etapa 5 do `plan.md`).

## 5. De penalidade para QUBO

Um **QUBO** (*Quadratic Unconstrained Binary Optimization*) é uma função da forma

$$
f(x) = x^T Q x + c^T x + d, \qquad x \in \{0,1\}^n, \quad \text{sem restrições.}
$$

É exatamente isso que um computador quântico adiabático ou um QAOA sabe minimizar. Para chegar lá, basta expandir a penalidade. Usamos que, para variáveis binárias, $x_i^2 = x_i$. Assim, o quadrado da soma pode ser escrito como uma forma quadrática, $(\mathbf{1}^T x)^2 = x^T(\mathbf{1}\mathbf{1}^T)x$:

$$
(\mathbf{1}^T x - B)^2 = x^T(\mathbf{1}\mathbf{1}^T)x - 2B\,\mathbf{1}^T x + B^2
$$

Juntando os termos:

$$
Q = q\,\Sigma + \lambda\,\mathbf{1}\mathbf{1}^T, \qquad
c = -\mu - 2\lambda B\,\mathbf{1}, \qquad
d = \lambda B^2
$$

O número de **qubits** é o tamanho de $x$: um qubit por ativo. Aqui, $n = 4$.

In [6]:
lam = 1.0
ones = np.ones(n)

Q = q * Sigma.values + lam * np.outer(ones, ones)   # termo quadrático
c = -mu.values - 2 * lam * B * ones                  # termo linear
d = lam * B ** 2                                      # constante

def f_qubo(x):
    return x @ Q @ x + c @ x + d

# Verificação: a forma matricial é idêntica à forma penalizada para os 16 vetores
erro = max(abs(f_qubo(x) - f_penalizada(x, lam)) for x in todos)
print(f'Erro máximo entre forma QUBO e forma penalizada: {erro:.2e}')
assert erro < 1e-12, 'a expansão do QUBO não bate com a penalidade'

print(f'\nNúmero de variáveis binárias (= qubits necessários): {n}')
print('\nMatriz Q:')
print(pd.DataFrame(Q, index=ativos, columns=ativos).round(4).to_string())
print('\nVetor linear c:', dict(zip(ativos, c.round(4).tolist())))
print('Constante d:', d)

Erro máximo entre forma QUBO e forma penalizada: 1.17e-15

Número de variáveis binárias (= qubits necessários): 4

Matriz Q:
          BBAS3.SA  ITUB4.SA  KLBN4.SA  VALE3.SA
BBAS3.SA    1.0221    1.0133    1.0036    1.0035
ITUB4.SA    1.0133    1.0221    1.0037    1.0052
KLBN4.SA    1.0036    1.0037    1.0225    1.0058
VALE3.SA    1.0035    1.0052    1.0058    1.0308

Vetor linear c: {'BBAS3.SA': -4.2946, 'ITUB4.SA': -4.1956, 'KLBN4.SA': -4.2037, 'VALE3.SA': -3.8669}
Constante d: 4.0


## 6. De QUBO para Ising

QAOA e VQE trabalham com operadores de Pauli. Para isso, trocamos as variáveis binárias por variáveis de spin, $z_i \in \{-1,+1\}$, com

$$
x_i = \frac{1 - z_i}{2}, \qquad \text{ou seja} \qquad z_i = 1 - 2x_i
$$

Substituindo em $f(x)$ e usando $z_i^2 = 1$, obtemos

$$
f(x) = \text{const} + \sum_i h_i z_i + \sum_{i<j} J_{ij} z_i z_j
$$

com $h_i$ e $J_{ij}$ determinados por $Q$ e $c$. Em operadores quânticos, $z_i$ vira $Z_i$ (Pauli-Z), e a soma vira o Hamiltoniano $H = \sum_i h_i Z_i + \sum_{i<j} J_{ij} Z_i Z_j$. O estado de menor energia de $H$ corresponde ao vetor ótimo $x^*$.

A função abaixo faz essa conversão e a checagem confirma que as duas formas dão o mesmo valor para os 16 vetores.

In [7]:
def qubo_para_ising(Q, c, d):
    """Converte f(x) = xᵀQx + cᵀx + d em E(z) = off + Σ h_i z_i + Σ_{i<j} J_ij z_i z_j."""
    n_ = len(c)
    h = np.zeros(n_)
    J = np.zeros((n_, n_))
    off = d
    for i in range(n_):
        off += c[i] / 2
        h[i] -= c[i] / 2
        for j in range(n_):
            off += Q[i, j] / 4
            h[i] -= Q[i, j] / 4
            h[j] -= Q[i, j] / 4
            if i == j:
                off += Q[i, j] / 4        # z_i² = 1 contribui com constante
            else:
                J[i, j] += Q[i, j] / 4
    return h, J, off

def energia_ising(z, h, J, off):
    e = off + h @ z
    for i in range(len(z)):
        for j in range(i + 1, len(z)):
            e += (J[i, j] + J[j, i]) * z[i] * z[j]
    return e

h, J, off = qubo_para_ising(Q, c, d)

erro_ising = max(abs(f_qubo(x) - energia_ising(1 - 2 * x, h, J, off)) for x in todos)
print(f'Erro máximo entre QUBO e Ising (com z = 1 − 2x): {erro_ising:.2e}')
assert erro_ising < 1e-12, 'a conversão para Ising não bate com o QUBO'

print('\nCampos locais h_i (um por qubit):', dict(zip(ativos, h.round(4).tolist())))
print('Acoplamentos J_ij (pares de qubits):')
pares = [(ativos[i], ativos[j], round(J[i, j] + J[j, i], 4)) for i in range(n) for j in range(i + 1, n)]
for a, b, v in pares:
    print(f'  {a} – {b}: {v}')
print(f'Constante: {off:.4f}')
print(f'\nHamiltoniano: {n} qubits, {n} termos Z, {len(pares)} termos ZZ.')

Erro máximo entre QUBO e Ising (com z = 1 − 2x): 1.78e-15

Campos locais h_i (um por qubit): {'BBAS3.SA': 0.1261, 'ITUB4.SA': 0.0757, 'KLBN4.SA': 0.0841, 'VALE3.SA': -0.0891}
Acoplamentos J_ij (pares de qubits):
  BBAS3.SA – ITUB4.SA: 0.5067
  BBAS3.SA – KLBN4.SA: 0.5018
  BBAS3.SA – VALE3.SA: 0.5017
  ITUB4.SA – KLBN4.SA: 0.5018
  ITUB4.SA – VALE3.SA: 0.5026
  KLBN4.SA – VALE3.SA: 0.5029
Constante: 0.7859

Hamiltoniano: 4 qubits, 4 termos Z, 6 termos ZZ.


## 7. Conferindo a solução pela força bruta

O mínimo do QUBO (e do Ising, que é o mesmo problema) deve ser a melhor carteira viável. Vamos conferir.

In [8]:
valores = [(x, f_qubo(x)) for x in todos]
x_ot, v_ot = min(valores, key=lambda t: t[1])

print('Mínimo global do QUBO:', x_ot, '| viável:', x_ot.sum() == B, '| valor:', round(v_ot, 4))
print('Ativos escolhidos:', [a for a, s in zip(ativos, x_ot) if s])

# O mesmo mínimo visto pelo Ising: qual z tem energia mínima?
z_ot = 1 - 2 * x_ot
print('Energia Ising do mesmo vetor:', round(energia_ising(z_ot, h, J, off), 4))

Mínimo global do QUBO: [1 0 1 0] | viável: True | valor: -0.4465
Ativos escolhidos: ['BBAS3.SA', 'KLBN4.SA']
Energia Ising do mesmo vetor: -0.4465


### Efeito do apetite ao risco $q$

Com $q$ pequeno, a carteira procura retorno; com $q$ grande, procura baixo risco. Vale ver como a escolha muda, considerando só as 6 carteiras viáveis e sem penalidade.

In [9]:
viaveis = [x for x in todos if x.sum() == B]

print(f'{"q":>6}  {"ativos escolhidos":<45}  f(x)')
for q_t in [0.1, 0.5, 2.0, 10.0]:
    best = min(viaveis, key=lambda x: objetivo(x, q_t))
    nomes = ', '.join(a for a, s in zip(ativos, best) if s)
    print(f'{q_t:>6}  {nomes:<45}  {objetivo(best, q_t):.4f}')

     q  ativos escolhidos                              f(x)
   0.1  BBAS3.SA, KLBN4.SA                             -0.4879
   0.5  BBAS3.SA, KLBN4.SA                             -0.4465
   2.0  BBAS3.SA, KLBN4.SA                             -0.2910
  10.0  BBAS3.SA, KLBN4.SA                             0.5384


**Leitura do resultado:** neste intervalo de $q$, a escolha não muda. Os dados explicam o motivo: BBAS3 tem o maior retorno esperado, KLBN4 tem retorno positivo e correlação baixa com os demais (0,16 com BBAS3 e ITUB4), enquanto BBAS3 e ITUB4 são muito correlacionados (0,60), e VALE3 tem retorno negativo. A penalidade de risco de $q$ não basta para trocar um par por outro. Isso mostra que a escolha depende dos dados de treino, e não só da formulação.

**Atenção à escala:** com $\lambda = 1$, a matriz $Q$ é dominada pelo termo $\lambda\,\mathbf{1}\mathbf{1}^T$ (cerca de 1,0 em cada entrada), enquanto $q\,\Sigma$ fica na casa de 0,01. Ou seja, a penalidade quase não deixa espaço para o risco entrar na decisão. Isso é uma consequência direta da escolha de $\lambda$ e deve ser considerado ao interpretar os resultados quânticos.

## 8. Como isso se conecta aos algoritmos quânticos (conceitual)

Até aqui não usamos nenhum algoritmo quântico. O que construímos é o que eles precisam receber:

- **QAOA:** monta um circuito de $p$ camadas. Cada camada aplica uma rotação pela parte de custo, $e^{-i\gamma H}$, e uma rotação de mistura, $e^{-i\beta \sum_i X_i}$. Os $2p$ ângulos $(\gamma, \beta)$ são otimizados por um método clássico, minimizando o valor esperado de $H$. Na DualQ, $p = 20$.
- **VQE:** usa um *ansatz* genérico (na DualQ, rotações RY com emaranhamento CZ) em vez de partir do próprio Hamiltoniano. O princípio variacional garante que o valor esperado nunca fica abaixo do menor autovalor.
- **Em ambos,** o resultado é uma distribuição sobre bitstrings. A resposta é a bitstring de maior probabilidade (ou de menor energia), e a verificação é checar se ela é viável.

## 9. O que esta formulação deixa em aberto

1. **Penalidade $\lambda$:** o limiar depende da escala de $\mu$ e $\Sigma$ (seção 4). Precisa ser escolhido com teste, e não por palpite.
2. **Seleção binária vs. pesos contínuos:** aqui só decidimos *quais* ativos entram, com pesos iguais. Markowitz contínuo decide *quanto* alocar em cada um. A cardinalidade fixa é a escolha do E-09 do `errors.md`.
3. **Qualidade de $\mu$:** com 2 anos de treino, $\mu$ é uma estimativa ruidosa. O QUBO herda esse ruído. O teste fora da amostra do notebook 01 (2025) já mostrou o efeito disso. Uma solução melhor no QUBO não significa uma carteira melhor no futuro.
4. **Escala do problema:** com 4 ativos, o mínimo sai da força bruta em microssegundos. A parte quântica só faz sentido de comparação metodológica até o problema crescer.

## 10. Próximo passo: mapear para as ferramentas atuais do Qiskit

**Ainda não verificado.** Antes de escrever código quântico, vamos conferir na documentação atual:

- como representar o QUBO acima (`QuadraticProgram` em `qiskit-optimization`) e a conversão para Ising (`to_ising`);
- se o módulo `qiskit.finance` da DualQ ainda existe ou foi substituído;
- qual versão de `qiskit`, `qiskit-algorithms` e `qiskit-aer` é compatível com a versão de Python escolhida para o `venv-quantum`.

A checagem desta seção (QUBO e Ising batendo com a força bruta nos 16 vetores) é o critério que qualquer implementação posterior precisa passar.